In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from scipy.sparse import coo_matrix
from sklearn.preprocessing import OneHotEncoder
import pickle

# -------------------------
# 1. DATA GENERATION
# -------------------------

np.random.seed(42)

NUM_USERS = 1000
NUM_ITEMS = 300
NUM_ORDERS = 15000

flavors = ["spicy", "sweet", "tangy", "savory"]
textures = ["crunchy", "liquid", "soft"]
temperatures = ["hot", "cold"]
categories = ["main", "side", "beverage", "dessert"]
heaviness = ["light", "heavy"]

items = []
for i in range(NUM_ITEMS):
    items.append([
        i,
        random.choice(flavors),
        random.choice(textures),
        random.choice(temperatures),
        random.choice(categories),
        random.choice(heaviness)
    ])

items_df = pd.DataFrame(items, columns=[
    "item_id","flavor","texture","temperature","category","heaviness"
])

orders = []
for _ in range(NUM_ORDERS):
    orders.append([
        random.randint(0, NUM_USERS-1),
        random.randint(0, NUM_ITEMS-1)
    ])

orders_df = pd.DataFrame(orders, columns=["user_id","item_id"])

# -------------------------
# 2. BUILD USER-ITEM GRAPH
# -------------------------

rows = orders_df.user_id.values
cols = orders_df.item_id.values

interaction = coo_matrix(
    (np.ones(len(rows)), (rows, cols)),
    shape=(NUM_USERS, NUM_ITEMS)
)

# Build symmetric adjacency
adj = coo_matrix(
    (np.ones(len(rows)*2),
     (np.concatenate([rows, cols+NUM_USERS]),
      np.concatenate([cols+NUM_USERS, rows]))),
    shape=(NUM_USERS+NUM_ITEMS, NUM_USERS+NUM_ITEMS)
)

adj = adj.tocsr()
rowsum = np.array(adj.sum(1))
d_inv = np.power(rowsum, -0.5).flatten()
d_inv[np.isinf(d_inv)] = 0.
d_mat_inv = coo_matrix((d_inv, (range(len(d_inv)), range(len(d_inv)))))
adj_norm = d_mat_inv.dot(adj).dot(d_mat_inv)
adj_norm = adj_norm.tocoo()

indices = torch.LongTensor([adj_norm.row, adj_norm.col])
values = torch.FloatTensor(adj_norm.data)
adj_norm = torch.sparse.FloatTensor(indices, values, torch.Size(adj_norm.shape))

# -------------------------
# 3. LIGHTGCN MODEL
# -------------------------

class LightGCN(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64, layers=3):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.layers = layers

        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self):
        users = self.user_emb.weight
        items = self.item_emb.weight
        all_emb = torch.cat([users, items])
        embs = [all_emb]

        for _ in range(self.layers):
            all_emb = torch.sparse.mm(adj_norm, all_emb)
            embs.append(all_emb)

        final_emb = torch.mean(torch.stack(embs), dim=0)
        users_final = final_emb[:self.num_users]
        items_final = final_emb[self.num_users:]
        return users_final, items_final

# -------------------------
# 4. BPR TRAINING LOOP
# -------------------------

model = LightGCN(NUM_USERS, NUM_ITEMS)
optimizer = optim.Adam(model.parameters(), lr=0.001)

def bpr_loss(u, pos, neg):
    pos_scores = torch.sum(u * pos, dim=1)
    neg_scores = torch.sum(u * neg, dim=1)
    return -torch.mean(torch.log(torch.sigmoid(pos_scores - neg_scores)))

for epoch in range(10):
    model.train()
    users_final, items_final = model()

    user_ids = torch.LongTensor(np.random.randint(0, NUM_USERS, 1024))
    pos_ids = torch.LongTensor(np.random.randint(0, NUM_ITEMS, 1024))
    neg_ids = torch.LongTensor(np.random.randint(0, NUM_ITEMS, 1024))

    u = users_final[user_ids]
    pos = items_final[pos_ids]
    neg = items_final[neg_ids]

    loss = bpr_loss(u, pos, neg)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch} Loss: {loss.item():.4f}")

torch.save(model.state_dict(), "lightgcn.pth")

# -------------------------
# 5. MSV VECTOR CREATION
# -------------------------

enc = OneHotEncoder()
food_vec = enc.fit_transform(
    items_df[["flavor","texture","temperature","category","heaviness"]]
).toarray()

ideal_meal = food_vec.mean(axis=0)

with open("msv_vectors.pkl","wb") as f:
    pickle.dump((food_vec, ideal_meal), f)

print("Training complete.")

/tmp/ipython-input-326/569866720.py:79: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  indices = torch.LongTensor([adj_norm.row, adj_norm.col])
/tmp/ipython-input-326/569866720.py:81: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  adj_norm = torch.sparse.FloatTensor(indices, values, torch.Size(adj_norm.shape))


Epoch 0 Loss: 0.6932
Epoch 1 Loss: 0.6931
Epoch 2 Loss: 0.6932
Epoch 3 Loss: 0.6931
Epoch 4 Loss: 0.6931
Epoch 5 Loss: 0.6932
Epoch 6 Loss: 0.6932
Epoch 7 Loss: 0.6932
Epoch 8 Loss: 0.6932
Epoch 9 Loss: 0.6931
Training complete.
